# MPIIGaze 시선 추정: CNN + Head Pose
눈 이미지 `(1, 36, 60)`와 머리 방향 `(yaw, pitch)`를 함께 사용해 시선의 두 각도(라디안)를 예측합니다.
`Image/`와 `Label/`은 각각 입력 이미지와 그 이미지에 대응하는 정답·머리 방향이며, 하나의 데이터셋을 이룹니다.

- 평가: **15-fold leave-one-person-out**. 매번 새 모델로 14명(42,000개)을 학습하고 남은 1명(3,000개)을 평가합니다.
- 최종 지표: **평균 각도 오차(°), 낮을수록 좋음**. 분류 문제의 정확도(%)를 사용하지 않습니다.
- 학습 설정과 epoch 수는 모든 fold에서 동일합니다. 테스트 성능으로 모델·epoch를 선택하지 않습니다.
- 이미지 CNN + head pose 분기, AdamW, 학습률 감소, Dropout, 밝기 증강, EMA를 사용합니다.
- 아래 셀을 위에서부터 실행하세요. **「전체 학습 실행」 셀부터 실제 학습이 시작됩니다.**
- 실제 성능은 실행 후 확인해야 합니다. README의 ridge **7.27°**는 비교 기준이며, 이를 이긴다고 보장하는 수치는 아닙니다.

수업의 Conv/ReLU/Pooling/Affine 개념을 PyTorch로 구현합니다. NumPy 수업 코드를 그대로 복사하지 않아도 되며, PyTorch가 역전파를 자동 계산합니다.


## 1. 실행 환경
Python 3.10 이상(권장 3.12)의 가상환경에서 Jupyter를 실행하는 것을 권장합니다.
이미 필요한 패키지가 설치되어 있다면 아래 설치 셀은 그대로 건너뜁니다.
`%pip`는 현재 노트북 커널에 설치합니다. 설치 후에는 커널을 재시작하세요.


In [ ]:
# 패키지가 없을 때만 맨 앞의 #을 지우고 실행하세요.
# %pip install torch numpy opencv-python matplotlib pandas tqdm


In [ ]:
from pathlib import Path
import copy
import gc
import hashlib
import json
import math
import os
import random
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from IPython.display import display

# Mac은 MPS, NVIDIA GPU는 CUDA, 가속기가 없으면 CPU를 사용합니다.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

cv2.setNumThreads(0)
print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| 학습 장치:", DEVICE)
if DEVICE.type == "cpu":
    print("CPU에서도 실행되지만 15개 모델의 학습에는 오래 걸릴 수 있습니다.")


## 2. 경로와 공통 설정
기본 데이터 경로는 `gaze_estimation/data/EyeBased/MPIIGaze`입니다.
노트북을 프로젝트 폴더 또는 그 상위 폴더에서 열어도 찾을 수 있게 설정했습니다.
다른 위치의 데이터를 쓰면 `DATA_ROOT_OVERRIDE`에 **Image와 Label을 함께 포함한 MPIIGaze 폴더**를 지정하세요.

기본값은 모든 fold를 각각 50 epoch 학습합니다. 학습 중 보는 값은 학습 손실뿐입니다.
검증/early stopping 없이 미리 고정한 학습 예산을 사용하므로, 14명 전부를 학습에 사용할 수 있습니다.
설정을 조정하려면 테스트 사람을 제외한 학습 사람들 안에서 별도의 사람 단위 검증을 먼저 설계해야 합니다.
테스트 결과를 본 뒤 설정을 바꾸고 같은 결과를 공정한 최종 성능으로 보고하면 안 됩니다.


In [ ]:
# 다른 디스크에 데이터가 있을 때만 수정하세요.
DATA_ROOT_OVERRIDE = None  # 예: "/Volumes/Data/EyeBased/MPIIGaze"

# reader.py가 있는 프로젝트 폴더를 찾습니다.
candidates = [Path.cwd(), Path.cwd() / "gaze_estimation", *Path.cwd().parents]
PROJECT_ROOT = next(
    (p.resolve() for p in candidates if (p / "reader.py").is_file()
     and (p / "preprocess" / "mpii_norm.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("gaze_estimation 폴더에서 Jupyter를 열고 다시 실행하세요.")
sys.path.insert(0, str(PROJECT_ROOT))

import reader
from preprocess.mpii_norm import gaze_to_3d

DATA_ROOT = (Path(DATA_ROOT_OVERRIDE).expanduser().resolve()
             if DATA_ROOT_OVERRIDE else PROJECT_ROOT / "data" / "EyeBased" / "MPIIGaze")
IMAGE_ROOT = DATA_ROOT / "Image"
LABEL_ROOT = DATA_ROOT / "Label"
if not IMAGE_ROOT.is_dir() or not LABEL_ROOT.is_dir():
    raise FileNotFoundError(
        f"Image/Label 폴더가 없습니다: {DATA_ROOT}\n"
        "프로젝트 폴더에서 python3 -m preprocess.mpii 를 실행하거나 DATA_ROOT_OVERRIDE를 확인하세요."
    )

PERSONS = [f"p{k:02d}" for k in range(15)]
FOLDS = PERSONS.copy()  # 첫 실행 점검만 할 때 ["p00"]. 최종 보고는 반드시 15명 전체.
CFG = {
    "seed": 42,
    "epochs": 50,
    "batch_size": 128,
    "learning_rate": 1e-3,
    "min_learning_rate": 1e-5,
    "warmup_epochs": 3,
    "weight_decay": 1e-3,
    "dropout": 0.35,
    "ema_decay": 0.99,
    "gradient_clip": 5.0,
    "photometric_augmentation": True,
}
# 같은 설정·데이터의 완료된 fold는 재사용합니다. 중단된 fold는 처음부터 다시 학습합니다.
RESUME_COMPLETED = True
OUTPUT_BASE = PROJECT_ROOT / "outputs" / "gaze_cnn"

assert CFG["epochs"] > CFG["warmup_epochs"] >= 0
assert CFG["batch_size"] > 0
assert len(FOLDS) == len(set(FOLDS)) and set(FOLDS).issubset(PERSONS)
print("프로젝트:", PROJECT_ROOT)
print("데이터:", DATA_ROOT)
print("평가 대상:", FOLDS)
print("전체 설정:", CFG)


## 3. 데이터 검사와 메모리 캐시
헤더 이름으로 `Image`, `2DHead`, `2DGaze`만 읽습니다. 다른 라벨 칸은 모델 입력에 사용하지 않습니다.
15명 모두 3,000개인지, 이미지 크기와 라벨 값이 올바른지 검사합니다.

45,000장의 작은 이미지를 한 번만 읽어 RAM에 uint8로 보관합니다(이미지 배열 약 97MB).
배치를 만들 때 float32/0~1로 변환하므로 매 epoch마다 JPEG를 다시 읽지 않습니다.
테스트 이미지를 RAM에 보관하는 것은 데이터 읽기일 뿐이며, 학습 배치와 통계에는 테스트 사람을 포함하지 않습니다.
오른눈은 이미 전처리되어 있으므로 추가 좌우 반전이나 회전 증강은 하지 않습니다.


In [ ]:
def load_data_cache(data_root):
    images, heads, gazes, person_ids, names = [], [], [], [], []
    fingerprint = hashlib.sha256()
    counts = []
    for person in tqdm(PERSONS, desc="이미지 검사 및 캐시"):
        label_file = data_root / "Label" / f"{person}.label"
        if not label_file.is_file():
            raise FileNotFoundError(label_file)
        fingerprint.update(label_file.read_bytes())
        rows = reader.read_label(str(label_file))
        if len(rows) != 3000:
            raise ValueError(f"{person}: 3,000개를 예상했지만 {len(rows)}개입니다.")
        seen_names = set()
        for name, gaze_text, head_text in rows:
            relative_path = Path(name)
            if (relative_path.is_absolute() or ".." in relative_path.parts
                    or relative_path.parts[0] != person or name in seen_names):
                raise ValueError(f"이미지 경로 또는 중복 파일명을 확인하세요: {name}")
            seen_names.add(name)
            path = data_root / "Image" / relative_path
            image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
            if image is None or image.shape != (36, 60):
                raise ValueError(f"읽기 실패 또는 잘못된 이미지 크기: {path}")
            head = np.array(head_text.split(","), dtype=np.float32)
            gaze = np.array(gaze_text.split(","), dtype=np.float32)
            if (head.shape != (2,) or gaze.shape != (2,)
                    or not np.isfinite(head).all() or not np.isfinite(gaze).all()):
                raise ValueError(f"잘못된 head/gaze 라벨: {name}")
            if abs(float(gaze[1])) > math.pi / 2 + 1e-5:
                raise ValueError(f"pitch 범위를 확인하세요. 라벨은 라디안이어야 합니다: {name}")
            images.append(image[None])
            heads.append(head)
            gazes.append(gaze)
            person_ids.append(person)
            names.append(name)
            fingerprint.update(name.encode("utf-8"))
            fingerprint.update(image.tobytes())
        counts.append({"person": person, "samples": len(rows)})
    cache = {
        "images": np.stack(images),  # (45000, 1, 36, 60), uint8
        "heads": np.stack(heads),    # (45000, 2), float32
        "gazes": np.stack(gazes),    # (45000, 2), float32
        "persons": np.array(person_ids),
        "names": np.array(names),
    }
    return cache, pd.DataFrame(counts), fingerprint.hexdigest()

CACHE, data_counts, DATA_HASH = load_data_cache(DATA_ROOT)
display(data_counts)
print("전체 샘플:", len(CACHE["names"]))
print(f"이미지 배열 RAM: {CACHE['images'].nbytes / 1e6:.1f} MB")
print("이미지:", CACHE["images"].shape, "| head:", CACHE["heads"].shape,
      "| gaze:", CACHE["gazes"].shape)

# 예제 하나를 눈으로 확인합니다. 아직 학습은 하지 않습니다.
plt.figure(figsize=(6, 2))
plt.imshow(CACHE["images"][0, 0], cmap="gray", vmin=0, vmax=255)
plt.title(f"{CACHE['names'][0]} | gaze(rad)={CACHE['gazes'][0].round(3)}")
plt.axis("off")
plt.show()


## 4. 평가 지표와 배치 로더
README와 동일하게 두 시선을 3D 단위 벡터로 바꾼 뒤 각도 차이를 도(°)로 계산합니다.
각 샘플의 오차를 모아 평균을 내므로, 마지막 배치가 작아도 평가에서 빠지지 않습니다.
Mac/Jupyter의 프로세스 생성 문제를 피하기 위해 `num_workers=0`을 사용합니다.


In [ ]:
def angular_errors_deg(pred, target):
    pred = np.asarray(pred, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    if pred.shape != target.shape or pred.ndim != 2 or pred.shape[1] != 2:
        raise ValueError("pred/target은 같은 (N, 2) 모양이어야 합니다.")
    a, b = gaze_to_3d(pred), gaze_to_3d(target)
    cosine = np.sum(a * b, axis=-1) / (
        np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1)
    )
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))


class EyeDataset(Dataset):
    def __init__(self, cache, indices):
        self.cache = cache
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        i = self.indices[item]
        return (
            torch.from_numpy(self.cache["images"][i]),
            torch.from_numpy(self.cache["heads"][i]),
            torch.from_numpy(self.cache["gazes"][i]),
        )


def make_loader(cache, indices, shuffle, seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        EyeDataset(cache, indices),
        batch_size=CFG["batch_size"],
        shuffle=shuffle,
        drop_last=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )


def to_device(batch):
    eye, head, gaze = batch
    eye = eye.to(DEVICE, dtype=torch.float32) / 255.0
    head = head.to(DEVICE, dtype=torch.float32)
    gaze = gaze.to(DEVICE, dtype=torch.float32)
    return eye, head, gaze


## 5. CNN과 머리 방향 결합 모델
이미지: `Conv → 정규화 → ReLU`를 두 번 적용한 뒤 MaxPool로 줄이는 블록 3개를 사용합니다.
공간 위치 정보가 남도록 마지막 특징 맵을 펼쳐 사용합니다.
머리 방향: 2개 각도를 작은 완전연결 계층으로 처리한 뒤 이미지 특징과 합칩니다.

GroupNorm은 샘플별로 채널 그룹을 정규화하므로 BatchNorm의 학습/추론 통계 관리가 필요 없습니다.
이미지는 항상 고정식 `(x - 0.5) / 0.5`로 변환합니다.
head의 평균·표준편차와 출력층의 초기 gaze 평균은 **현재 fold의 학습용 14명만**으로 계산합니다.
출력층에 Softmax를 붙이지 않고 라디안 두 값을 그대로 예측합니다.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class GazeCNN(nn.Module):
    def __init__(self, head_mean, head_std, gaze_mean, dropout=0.35):
        super().__init__()
        # buffer도 state_dict에 저장되어 추론 시 같은 정규화를 사용합니다.
        self.register_buffer("head_mean", torch.as_tensor(head_mean, dtype=torch.float32))
        self.register_buffer("head_std", torch.as_tensor(head_std, dtype=torch.float32))
        self.eye_features = nn.Sequential(
            ConvBlock(1, 24),   # 36x60 -> 18x30
            ConvBlock(24, 48),  # 18x30 -> 9x15
            ConvBlock(48, 96),  # 9x15 -> 4x7
            nn.Flatten(),
            nn.Linear(96 * 4 * 7, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head_features = nn.Sequential(nn.Linear(2, 32), nn.ReLU())
        self.regressor = nn.Sequential(
            nn.Linear(256 + 32, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )
        # ReLU 계층은 He 초기화, 마지막 출력은 작은 값에서 시작합니다.
        for layer in self.modules():
            if isinstance(layer, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
        nn.init.normal_(self.regressor[-1].weight, std=0.001)
        with torch.no_grad():
            self.regressor[-1].bias.copy_(torch.as_tensor(gaze_mean, dtype=torch.float32))

    def forward(self, eye, head):
        image_feature = self.eye_features((eye - 0.5) / 0.5)
        head_feature = self.head_features((head - self.head_mean) / self.head_std)
        return self.regressor(torch.cat([image_feature, head_feature], dim=1))


## 6. 학습 보조 함수
학습 배치에만 약한 밝기·대비·감마·노이즈 증강을 적용해 조명 변화에 대응합니다.
기하학적 변환은 이미지와 시선 방향의 관계를 바꿀 수 있어 적용하지 않습니다.

AdamW로 가중치를 갱신하고, 초반에는 학습률을 서서히 올린 뒤 cosine 방식으로 줄입니다.
EMA는 학습 중 가중치의 이동평균을 유지하는 기법입니다. 모든 fold에서 **마지막 epoch의 EMA 모델**로 평가합니다.
테스트 오차를 기준으로 'best model'을 고르지 않습니다.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    if DEVICE.type == "mps":
        torch.mps.manual_seed(seed)
    # seed는 고정하지만 장치/라이브러리 버전에 따른 수치 차이는 있을 수 있습니다.


def augment_eye(eye):
    # 샘플별로 조명만 조금 바꾸며, gaze/head 라벨은 유지합니다.
    n = eye.shape[0]
    shape = (n, 1, 1, 1)
    contrast = 0.85 + 0.30 * torch.rand(shape, device=eye.device)
    brightness = 0.10 * (torch.rand(shape, device=eye.device) - 0.5)
    gamma = 0.85 + 0.30 * torch.rand(shape, device=eye.device)
    mean = eye.mean(dim=(2, 3), keepdim=True)
    augmented = ((eye - mean) * contrast + mean + brightness).clamp(0, 1)
    augmented = augmented.clamp_min(1e-6).pow(gamma)
    augmented = (augmented + 0.015 * torch.randn_like(augmented)).clamp(0, 1)
    apply_mask = torch.rand(shape, device=eye.device) < 0.8
    return torch.where(apply_mask, augmented, eye)


def learning_rate_at(epoch):
    warmup = CFG["warmup_epochs"]
    if epoch < warmup:
        return CFG["learning_rate"] * (epoch + 1) / max(warmup, 1)
    progress = (epoch - warmup) / max(CFG["epochs"] - warmup - 1, 1)
    return CFG["min_learning_rate"] + 0.5 * (
        CFG["learning_rate"] - CFG["min_learning_rate"]
    ) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def update_ema(ema_model, model, update_count):
    # 학습 초기의 무작위 가중치가 오래 남지 않도록 처음에는 감쇠를 작게 합니다.
    decay = min(CFG["ema_decay"], (1 + update_count) / (10 + update_count))
    for ema_parameter, parameter in zip(ema_model.parameters(), model.parameters()):
        ema_parameter.mul_(decay).add_(parameter, alpha=1 - decay)
    for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
        ema_buffer.copy_(buffer)


def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")
    temporary.replace(path)


def atomic_csv(path, frame):
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def release_device_memory():
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()


## 7. 한 fold 학습
이 함수는 **학습용 사람 목록만** 받습니다. 테스트 배치나 테스트 손실을 읽지 않습니다.
학습 중 기록하는 MSE는 증강과 Dropout이 적용된 배치의 손실입니다.
epoch가 끝날 때마다 기록을 저장하고, 정해진 50 epoch가 끝나면 EMA 가중치를 저장합니다.
중단된 fold의 optimizer 상태를 복원하는 기능은 없으며, 해당 fold는 새 모델로 재시작합니다.


In [ ]:
def train_fold(train_people, cache, fold_dir, signature):
    train_indices = np.flatnonzero(np.isin(cache["persons"], train_people))
    assert len(train_people) == 14 and len(set(train_people)) == 14
    assert len(train_indices) == 42000
    seed_everything(CFG["seed"])

    # 통계에는 학습 사람만 포함합니다.
    head_mean = cache["heads"][train_indices].mean(axis=0)
    head_std = np.maximum(cache["heads"][train_indices].std(axis=0), 1e-3)
    gaze_mean = cache["gazes"][train_indices].mean(axis=0)
    model = GazeCNN(head_mean, head_std, gaze_mean, CFG["dropout"]).to(DEVICE)
    ema_model = copy.deepcopy(model).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"]
    )
    loss_function = nn.MSELoss()
    train_loader = make_loader(cache, train_indices, shuffle=True, seed=CFG["seed"])
    history = []
    updates = 0
    start_time = time.perf_counter()

    for epoch in range(CFG["epochs"]):
        epoch_start = time.perf_counter()
        lr = learning_rate_at(epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        model.train()
        weighted_loss = torch.zeros((), device=DEVICE)
        sample_count = 0
        progress = tqdm(
            train_loader, desc=f"학습 {epoch + 1:02d}/{CFG['epochs']}", leave=False
        )
        for batch in progress:
            eye, head, target = to_device(batch)
            if CFG["photometric_augmentation"]:
                eye = augment_eye(eye)
            optimizer.zero_grad(set_to_none=True)
            prediction = model(eye, head)
            loss = loss_function(prediction, target)
            if not torch.isfinite(loss).item():
                raise FloatingPointError("손실이 NaN/Inf입니다. 입력 및 학습률을 확인하세요.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG["gradient_clip"])
            optimizer.step()
            updates += 1
            update_ema(ema_model, model, updates)
            weighted_loss += loss.detach() * len(target)
            sample_count += len(target)

        assert sample_count == len(train_indices)
        train_loss = weighted_loss.item() / sample_count
        history.append({
            "epoch": epoch + 1,
            "train_mse_rad2": train_loss,
            "learning_rate": lr,
            "epoch_seconds": time.perf_counter() - epoch_start,
        })
        atomic_csv(fold_dir / "history.csv", pd.DataFrame(history))
        print(f"  epoch {epoch + 1:02d}/{CFG['epochs']} | "
              f"train MSE={train_loss:.6f} | lr={lr:.2e}")

    # 학습 종료 모델만 저장합니다. 정규화 통계도 state_dict에 포함됩니다.
    checkpoint = {
        "state_dict": {key: value.detach().cpu().clone()
                       for key, value in ema_model.state_dict().items()},
        "config": dict(CFG),
        "train_people": list(train_people),
        "head_mean": head_mean.tolist(),
        "head_std": head_std.tolist(),
        "gaze_mean": gaze_mean.tolist(),
        "epochs_completed": CFG["epochs"],
        "signature": signature,
        "model_type": "final_epoch_ema",
    }
    checkpoint_path = fold_dir / "model.pt"
    temporary = fold_dir / "model.pt.tmp"
    torch.save(checkpoint, temporary)
    temporary.replace(checkpoint_path)
    duration = time.perf_counter() - start_time
    del model, ema_model, optimizer, train_loader
    release_device_memory()
    return checkpoint, duration


## 8. 학습 완료 모델 평가
추론에서는 Dropout과 데이터 증강을 끕니다.
각 테스트 샘플의 예측·정답·각도 오차를 CSV로 저장합니다.
학습 사람들의 평균 gaze만 예측하는 단순 기준도 함께 계산합니다(테스트 정답의 평균을 사용하지 않습니다).


In [ ]:
@torch.inference_mode()
def evaluate_checkpoint(checkpoint, cache, test_indices):
    model = GazeCNN(
        checkpoint["head_mean"], checkpoint["head_std"], checkpoint["gaze_mean"],
        checkpoint["config"]["dropout"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    model.eval()
    test_loader = make_loader(cache, test_indices, shuffle=False, seed=CFG["seed"])
    predictions, targets = [], []
    for batch in test_loader:
        eye, head, target = to_device(batch)
        predictions.append(model(eye, head).cpu().numpy())
        targets.append(target.cpu().numpy())
    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)
    if len(predictions) != len(test_indices) or not np.isfinite(predictions).all():
        raise ValueError("평가 샘플 수 또는 예측값을 확인하세요.")
    errors = angular_errors_deg(predictions, targets)
    naive_predictions = np.broadcast_to(
        np.array(checkpoint["gaze_mean"]), targets.shape
    )
    naive_error = angular_errors_deg(naive_predictions, targets).mean()
    frame = pd.DataFrame({
        "image": cache["names"][test_indices],
        "pred_yaw_rad": predictions[:, 0],
        "pred_pitch_rad": predictions[:, 1],
        "target_yaw_rad": targets[:, 0],
        "target_pitch_rad": targets[:, 1],
        "angular_error_deg": errors,
    })
    del model, test_loader
    release_device_memory()
    return frame, float(naive_error)


## 9. 전체 학습 실행 — 이 셀부터 실제 학습
기본값은 15개 fold를 차례로 학습하므로 한 모델 학습보다 시간이 많이 걸립니다.
학습 시간은 Mac 사양과 장치에 따라 달라지며, 완료 시간이나 성능을 미리 보장하지 않습니다.

실험별 결과는 `outputs/gaze_cnn/<실험 ID>/` 아래에 저장됩니다.
설정·데이터 내용·구현 코드가 달라지면 실험 ID가 달라져 이전 결과와 섞이지 않습니다.
모델이나 학습 함수 셀을 수정했다면 먼저 노트북을 저장하고 커널을 재시작해 위에서부터 실행하세요. 구현 식별자는 저장된 노트북 소스를 기준으로 계산합니다.
같은 실험에서 이 셀을 다시 실행하면 체크포인트와 평가 파일이 모두 있는 완료 fold를 건너뜁니다.
커널을 재시작했다면 위의 준비 셀부터 다시 실행하세요.

최종 모델은 매 fold에서 새로 초기화합니다. 이전 사람의 모델을 다음 fold 학습에 이어 쓰지 않습니다.


In [ ]:
def experiment_signature():
    # 실행 출력이 아닌, 구현 함수가 있는 셀의 소스만 식별자에 포함합니다.
    notebook_path = PROJECT_ROOT / "train.ipynb"
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    implementation = "\n".join(
        "".join(cell["source"]) for cell in notebook["cells"]
        if "implementation" in cell.get("metadata", {}).get("tags", [])
    )
    spec = {
        "protocol": "15_fold_lopo_fixed_epochs_final_ema_v1",
        "config": dict(CFG),
        "data_hash": DATA_HASH,
        "implementation_hash": hashlib.sha256(implementation.encode()).hexdigest(),
        "torch_version": str(torch.__version__),
        "numpy_version": np.__version__,
        "opencv_version": cv2.__version__,
        "device": str(DEVICE),
        "persons": PERSONS,
    }
    signature = hashlib.sha256(
        json.dumps(spec, sort_keys=True).encode("utf-8")
    ).hexdigest()
    return signature, spec


def run_lopo():
    signature, spec = experiment_signature()
    run_dir = OUTPUT_BASE / signature[:16]
    run_dir.mkdir(parents=True, exist_ok=True)
    atomic_json(run_dir / "experiment.json", spec)
    print("결과 저장 폴더:", run_dir)
    rows = []

    for test_person in FOLDS:
        train_people = [p for p in PERSONS if p != test_person]
        test_indices = np.flatnonzero(CACHE["persons"] == test_person)
        train_indices = np.flatnonzero(np.isin(CACHE["persons"], train_people))
        assert len(test_indices) == 3000 and len(train_indices) == 42000
        assert np.intersect1d(train_indices, test_indices).size == 0
        fold_dir = run_dir / test_person
        fold_dir.mkdir(exist_ok=True)
        result_path = fold_dir / "result.json"
        completed_files = ["model.pt", "history.csv", "predictions.csv", "result.json"]
        complete = all((fold_dir / name).is_file() for name in completed_files)

        if RESUME_COMPLETED and complete:
            row = json.loads(result_path.read_text(encoding="utf-8"))
            saved_history = pd.read_csv(fold_dir / "history.csv")
            saved_predictions = pd.read_csv(fold_dir / "predictions.csv")
            if (row["signature"] != signature or row["test_person"] != test_person
                    or row["train_people"] != train_people
                    or len(saved_history) != CFG["epochs"]
                    or len(saved_predictions) != len(test_indices)
                    or not np.isfinite(saved_predictions["angular_error_deg"]).all()
                    or saved_predictions["image"].tolist() != CACHE["names"][test_indices].tolist()
                    or not math.isclose(
                        row["mean_angular_error_deg"],
                        saved_predictions["angular_error_deg"].mean(), abs_tol=1e-9,
                    )):
                raise ValueError(f"저장된 결과와 현재 실험이 일치하지 않습니다: {fold_dir}")
            print(f"[재사용] {test_person}: {row['mean_angular_error_deg']:.3f}°")
        else:
            # 재학습 도중 중단되면 이전 완료 표시가 남지 않게 합니다.
            if result_path.exists():
                result_path.unlink()
            print(f"\n[{test_person}] 학습 14명 / 42,000개 → 테스트 1명 / 3,000개")
            checkpoint, seconds = train_fold(train_people, CACHE, fold_dir, signature)
            assert test_person not in checkpoint["train_people"]
            frame, naive_error = evaluate_checkpoint(checkpoint, CACHE, test_indices)
            atomic_csv(fold_dir / "predictions.csv", frame)
            row = {
                "test_person": test_person,
                "train_people": train_people,
                "n_train": len(train_indices),
                "n_test": len(test_indices),
                "mean_angular_error_deg": float(frame["angular_error_deg"].mean()),
                "median_angular_error_deg": float(frame["angular_error_deg"].median()),
                "naive_mean_gaze_error_deg": naive_error,
                "epochs": CFG["epochs"],
                "train_seconds": seconds,
                "signature": signature,
            }
            # 이 파일은 학습·평가 결과가 모두 저장된 뒤에만 생성합니다.
            atomic_json(result_path, row)
            print(f"[완료] {test_person}: {row['mean_angular_error_deg']:.3f}°")
            del checkpoint
        rows.append(row)
        atomic_csv(run_dir / "fold_results.csv", pd.DataFrame(rows))

    return pd.DataFrame(rows), run_dir


results, RUN_DIR = run_lopo()


## 10. 15-fold 결과표와 학습 곡선
전체 15명의 평균만 최종 LOPO 성능입니다. `FOLDS = ["p00"]`처럼 일부만 실행했다면 부분 결과로 표시합니다.
README의 7.27°는 기존 ridge 결과이며 아래 점선으로 표시합니다.
학습 손실이 낮아져도 새로운 사람의 오차가 반드시 낮아지는 것은 아닙니다.


In [ ]:
ordered = results.set_index("test_person").reindex(FOLDS)
display(ordered[[
    "n_train", "n_test", "mean_angular_error_deg",
    "median_angular_error_deg", "naive_mean_gaze_error_deg", "train_seconds",
]].round(4))

all_folds_done = set(results["test_person"]) == set(PERSONS) and len(results) == 15
mean_error = float(results["mean_angular_error_deg"].mean())
weighted_error = float(np.average(
    results["mean_angular_error_deg"], weights=results["n_test"]
))
label = "최종 15-fold 평균" if all_folds_done else "부분 fold 평균 (최종 성능 아님)"
print(f"{label}: {mean_error:.3f}°")
print(f"전체 평가 샘플 수: {int(results['n_test'].sum()):,}")
if all_folds_done:
    print(f"사람별 오차의 표준편차: {results['mean_angular_error_deg'].std(ddof=0):.3f}°")
    print(f"샘플별 가중 평균: {weighted_error:.3f}° (각 사람의 샘플 수가 같으므로 동일)")
    table = pd.DataFrame(
        [[*ordered.reindex(PERSONS)["mean_angular_error_deg"].values, mean_error]],
        columns=PERSONS + ["평균"], index=["CNN + head (°)"],
    )
    display(table.round(3))
    table.to_csv(RUN_DIR / "report_lopo.csv", encoding="utf-8-sig")
atomic_json(RUN_DIR / "summary.json", {
    "complete_15_fold": all_folds_done,
    "evaluated_persons": list(results["test_person"]),
    "mean_angular_error_deg": mean_error,
    "weighted_mean_angular_error_deg": weighted_error,
    "readme_ridge_reference_deg": 7.27,
})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(ordered.index, ordered["mean_angular_error_deg"], color="steelblue")
axes[0].axhline(7.27, color="tomato", linestyle="--", label="README ridge mean: 7.27 deg")
axes[0].set_ylabel("Mean angular error (degrees; lower is better)")
axes[0].set_xlabel("Held-out person")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend()
for person in FOLDS:
    history = pd.read_csv(RUN_DIR / person / "history.csv")
    axes[1].plot(history["epoch"], history["train_mse_rad2"], label=person, alpha=0.75)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training MSE (rad^2, augmented batches)")
axes[1].legend(ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(RUN_DIR / "results.png", dpi=150, bbox_inches="tight")
plt.show()


## 11. 저장된 모델로 샘플 예측 확인 (선택)
모델 저장 파일에는 가중치와 head 정규화 값이 함께 들어 있습니다.
아래 예시는 선택한 fold에서 제외했던 사람의 샘플을 그 fold 모델로 예측합니다.
15개 fold 모델은 서로 다른 학습 사람을 사용하므로, 서로의 테스트 예측을 섞어 평균내면 안 됩니다.
그렇게 하면 해당 사람을 학습한 모델이 평가에 섞입니다.

이 셀은 결과 확인용이며, 출력값을 보고 학습 설정을 바꾸는 용도가 아닙니다.


In [ ]:
SHOW_PERSON = FOLDS[0]
SHOW_INDEX = 0
checkpoint = torch.load(
    RUN_DIR / SHOW_PERSON / "model.pt", map_location="cpu", weights_only=True
)
assert SHOW_PERSON not in checkpoint["train_people"]
model = GazeCNN(
    checkpoint["head_mean"], checkpoint["head_std"], checkpoint["gaze_mean"],
    checkpoint["config"]["dropout"],
)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
indices = np.flatnonzero(CACHE["persons"] == SHOW_PERSON)
index = indices[SHOW_INDEX]
eye = torch.from_numpy(CACHE["images"][index:index + 1]).float() / 255.0
head = torch.from_numpy(CACHE["heads"][index:index + 1])
with torch.inference_mode():
    prediction = model(eye, head).numpy()
target = CACHE["gazes"][index:index + 1]
print("파일:", CACHE["names"][index])
print("예측 (yaw, pitch), rad:", prediction[0])
print("정답 (yaw, pitch), rad:", target[0])
print(f"각도 오차: {angular_errors_deg(prediction, target)[0]:.3f}°")
del model, checkpoint


## 참고와 결과 해석
- 과제 명세와 평가식: 이 저장소의 [README.md](README.md)
- 입력을 읽는 방법: [reader.py](reader.py), 데이터 확인: [read_sample.ipynb](read_sample.ipynb)
- 수업 코드의 Conv/ReLU/Pooling/완전연결 계층과 학습 루프를 바탕으로 이미지·head 두 입력을 결합했습니다.
- 논문의 multimodal CNN 개념을 참고한 확장 모델이며, 논문 구조의 정확한 재현은 아닙니다.
- [PyTorch MPS 안내](https://docs.pytorch.org/docs/stable/notes/mps.html), [GroupNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.GroupNorm.html), [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
- 좋은 성능을 목표로 한 초기 설정입니다. 측정 전에는 baseline 개선 여부를 알 수 없습니다.
- 제출할 때는 15명 각각의 각도 오차와 평균, 공통 설정, 사용 모델과 평가 절차를 함께 기록하세요.
- 이 노트북은 평가용으로 15개 모델을 만듭니다. 전체 15명으로 학습한 별도의 배포용 모델은 LOPO 평가 모델과 구분해야 합니다.
